In [135]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

In [136]:
directory_path = Path("final_clean_data")
files_list = [entry.name for entry in directory_path.iterdir() if entry.is_file()]

In [137]:
df = pd.read_csv("final_clean_data/"+files_list[0], index_col=0)

In [138]:
df = df.copy()

df['ret_1m'] = np.log(df['price'] / df['price'].shift(1))
df['ret_3m'] = np.log(df['price'] / df['price'].shift(3))

df['mom_12m'] = df['price'] / df['price'].shift(12) - 1

df['vol_3m'] = df['ret_1m'].rolling(3).std()
df['vol_6m'] = df['ret_1m'].rolling(6).std()

df['vol_z'] = (
    (df['volume'] - df['volume'].rolling(12).mean()) /
    df['volume'].rolling(12).std()
)

df['vol_trend'] = np.log(df['volume'] / df['volume'].shift(3))

df['ey'] = 1 / df['pe']

df['ey_z'] = (
    (df['ey'] - df['ey'].rolling(24).mean()) /
    df['ey'].rolling(24).std()
)

df['bm_z'] = (
    (df['bm'] - df['bm'].rolling(24).mean()) /
    df['bm'].rolling(24).std()
)

df['log_mc'] = np.log(df['mc'])

df['mc_z'] = (
    (df['log_mc'] - df['log_mc'].rolling(24).mean()) /
    df['log_mc'].rolling(24).std()
)

df['ma_6m'] = df['price'].rolling(6).mean()
df['rev_6m'] = (df['price'] - df['ma_6m']) / df['ma_6m']

df['mom_vol_adj'] = df['mom_12m'] / (df['vol_6m'] + 1e-8)
df['value_mom'] = df['ey'] * df['mom_12m']
df['liquidity_adj_mom'] = df['mom_12m'] * df['vol_z']

features = [
    'ret_1m',
    'ret_3m',
    'mom_12m',
    'vol_3m',
    'vol_6m',
    'vol_z',
    'vol_trend',
    'ey_z',
    'bm_z',
    'mc_z',
    'rev_6m',
    'mom_vol_adj',
    'value_mom',
    'liquidity_adj_mom'
]

rolling_mean = df[features].rolling(24).mean().shift(1)
rolling_std  = df[features].rolling(24).std().shift(1)

df[features] = (df[features] - rolling_mean) / (rolling_std + 1e-8)
df['R'] = np.log(df['price'].shift(-1) / df['price'])
features.append('R')

df = df[features]
df = df.dropna()

In [139]:
df

,ret_1m,ret_3m,mom_12m,vol_3m,vol_6m,vol_z,vol_trend,ey_z,bm_z,mc_z,rev_6m,mom_vol_adj,value_mom,liquidity_adj_mom,R
MOFS IN Equity,,,,,,,,,,,,,,,
2011-08-31,-1.177504,-1.193008,-1.149551,0.202303,0.689275,0.136251,0.537110,1.489090,-1.593914,-2.092554,-1.687181,-0.991727,-1.733234,0.383884,-0.042418
2011-09-30,-0.143511,-0.466204,-1.205884,-0.066388,0.759932,-0.077553,0.155569,1.337038,-1.320805,-1.661634,-1.382644,-1.047087,-1.913943,0.556420,0.029266
2011-10-31,0.534151,-0.599776,-1.144225,-0.375010,0.397379,3.297720,1.328401,0.752533,-0.994007,-1.207669,-0.264784,-1.102265,-1.730588,-1.667763,-0.024409
2011-11-30,-0.000049,0.286968,-0.963355,-1.341636,-0.570655,-1.264785,-1.457364,0.659091,-0.831514,-1.021665,-0.037527,-1.228425,-1.411880,1.460500,-0.059898
2011-12-31,-0.277209,0.205119,-0.989053,-1.092263,-0.887434,-0.723129,-0.505183,0.861412,-0.749540,-0.940111,-0.233278,-1.411747,-1.528277,1.075467,0.338088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-31,0.205907,-0.399393,-1.340016,-0.755041,-1.032223,-0.893692,-0.413445,-0.079029,-0.531272,-0.634202,-0.294132,-1.143594,-1.196107,-0.438492,-0.019149
2025-11-30,-0.486871,-0.215396,-1.289033,-1.020339,-1.656663,-1.322687,0.349552,-0.000717,-0.706329,-0.724025,-0.530761,-1.058017,-1.152977,-0.426481,-0.114382
2025-12-31,-1.068630,-0.792210,-1.366676,-0.361169,-1.251849,-1.051769,0.465041,0.548056,-1.332574,-1.255784,-1.069781,-1.223616,-1.209017,-0.291110,-0.127183


In [140]:
def get_normalised_data(file):
    df = pd.read_csv("final_clean_data/"+file, index_col=0)

    df = df.copy()
    
    df['ret_1m'] = np.log(df['price'] / df['price'].shift(1))
    df['ret_3m'] = np.log(df['price'] / df['price'].shift(3))
    
    df['mom_12m'] = df['price'] / df['price'].shift(12) - 1
    
    df['vol_3m'] = df['ret_1m'].rolling(3).std()
    df['vol_6m'] = df['ret_1m'].rolling(6).std()
    
    df['vol_z'] = (
        (df['volume'] - df['volume'].rolling(12).mean()) /
        df['volume'].rolling(12).std()
    )
    
    df['vol_trend'] = np.log(df['volume'] / df['volume'].shift(3))
    
    df['ey'] = 1 / df['pe']
    
    df['ey_z'] = (
        (df['ey'] - df['ey'].rolling(24).mean()) /
        df['ey'].rolling(24).std()
    )
    
    df['bm_z'] = (
        (df['bm'] - df['bm'].rolling(24).mean()) /
        df['bm'].rolling(24).std()
    )
    
    df['log_mc'] = np.log(df['mc'])
    
    df['mc_z'] = (
        (df['log_mc'] - df['log_mc'].rolling(24).mean()) /
        df['log_mc'].rolling(24).std()
    )
    
    df['ma_6m'] = df['price'].rolling(6).mean()
    df['rev_6m'] = (df['price'] - df['ma_6m']) / df['ma_6m']
    
    df['mom_vol_adj'] = df['mom_12m'] / (df['vol_6m'] + 1e-8)
    df['value_mom'] = df['ey'] * df['mom_12m']
    df['liquidity_adj_mom'] = df['mom_12m'] * df['vol_z']
    
    features = [
        'ret_1m',
        'ret_3m',
        'mom_12m',
        'vol_3m',
        'vol_6m',
        'vol_z',
        'vol_trend',
        'ey_z',
        'bm_z',
        'mc_z',
        'rev_6m',
        'mom_vol_adj',
        'value_mom',
        'liquidity_adj_mom'
    ]
    
    rolling_mean = df[features].rolling(24).mean().shift(1)
    rolling_std  = df[features].rolling(24).std().shift(1)
    
    df[features] = (df[features] - rolling_mean) / (rolling_std + 1e-8)
    df['R'] = np.log(df['price'].shift(-1) / df['price'])
    features.append('R')
    
    df = df[features]
    df = df.dropna()
    return df

In [159]:
def get_months():
    most_data_file = -1
    most_data = 0
    for file in files_list:
        data = get_normalised_data(file).shape[0]
        if data > most_data:
            most_data = data
            most_data_file = file
    df = get_normalised_data(most_data_file)
    months = list(df.index)
    return months

In [ ]:
df = df.copy()

df['ret_1m'] = np.log(df['price'] / df['price'].shift(1))
df['ret_3m'] = np.log(df['price'] / df['price'].shift(3))

df['mom_12m'] = df['price'] / df['price'].shift(12) - 1

df['vol_3m'] = df['ret_1m'].rolling(3).std()
df['vol_6m'] = df['ret_1m'].rolling(6).std()

df['vol_z'] = (
    (df['volume'] - df['volume'].rolling(12).mean()) /
    df['volume'].rolling(12).std()
)

df['vol_trend'] = np.log(df['volume'] / df['volume'].shift(3))

df['ey'] = 1 / df['pe']

df['ey_z'] = (
    (df['ey'] - df['ey'].rolling(24).mean()) /
    df['ey'].rolling(24).std()
)

df['bm_z'] = (
    (df['bm'] - df['bm'].rolling(24).mean()) /
    df['bm'].rolling(24).std()
)

df['log_mc'] = np.log(df['mc'])

df['mc_z'] = (
    (df['log_mc'] - df['log_mc'].rolling(24).mean()) /
    df['log_mc'].rolling(24).std()
)

df['ma_6m'] = df['price'].rolling(6).mean()
df['rev_6m'] = (df['price'] - df['ma_6m']) / df['ma_6m']

df['mom_vol_adj'] = df['mom_12m'] / (df['vol_6m'] + 1e-8)
df['value_mom'] = df['ey'] * df['mom_12m']
df['liquidity_adj_mom'] = df['mom_12m'] * df['vol_z']

features = [
    'ret_1m',
    'ret_3m',
    'mom_12m',
    'vol_3m',
    'vol_6m',
    'vol_z',
    'vol_trend',
    'ey_z',
    'bm_z',
    'mc_z',
    'rev_6m',
    'mom_vol_adj',
    'value_mom',
    'liquidity_adj_mom'
]

rolling_mean = df[features].rolling(24).mean().shift(1)
rolling_std  = df[features].rolling(24).std().shift(1)

df[features] = (df[features] - rolling_mean) / (rolling_std + 1e-8)
df['R'] = np.log(df['price'].shift(-1) / df['price'])
features.append('R')

df = df[features]
df = df.dropna()